# Step 7 — Optimal-Policy Model

**What this step answers:** what is the ideal parameter set per SKU to sit on the optimal side of total cost across the four legs — and how do you set that policy for a SKU you cannot simulate?

**Two parts, and the order matters.**

1. **The sweep (deterministic, no ML).** `PolicyOptimiser` evaluates the cost surface over inventory cover × service target for every SKU on the line and picks each SKU's minimum. *This is the operational answer.*
2. **The model (ML).** `PolicyModel` fits SKU **attributes** to those optimal parameters. This does not improve the answer — it generalises it, to SKUs the engine cannot sweep.

**Why not learn cost directly?** The engine already computes conversion cost causally from run hours, changeovers, overtime and absorption. A model fitted to engine output would be learning the engine — circular, and §9 territory. Step 7 was redefined at **D-064** for exactly this reason; the architecture is amended to v4.

**The use case that justifies the ML.** A **new launch** has no demand history, so Step 5a cannot characterise it and the engine cannot simulate it — but every attribute the policy model needs is known before first shipment. Hence two feature sets, fitted and reported separately: `FULL` and `LAUNCH`, the latter excluding `irreducible_volatility_cv` and `chronic_bias_l1`.

**For the pitch:** *the optimisation is deterministic; machine learning explains and generalises the policy rather than producing it.*

Prerequisite: Step 6 gate passed (D-051, restated at D-060).

## Setup

In [ ]:
import subprocess, os, sys

def sh(cmd, cwd=None):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    print(r.stdout[-3000:])
    if r.returncode != 0:
        print("STDERR:", r.stderr[-1500:])
    return r

REPO = '/content/ibp-tradeoff'
os.chdir('/content')
sh('rm -rf ibp-tradeoff')
sh('git clone https://github.com/rdelolmog-creator/ibp-tradeoff.git')
os.chdir(REPO); sys.path.insert(0, REPO)
sh('pip install -q xgboost shap scikit-learn')
print('cwd:', os.getcwd())

## Rebuild the Step 4 artefacts
Same approved pipeline, no code change.

In [ ]:
import pandas as pd, numpy as np, yaml
from src.ingest import DataIngestor
from src.cleaner import DataCleaner

if not os.path.isdir('data_primary/raw'):
    sh('python generate_data.py'); sh('mv data data_primary')

ing = DataIngestor(repo_root=REPO, data_root='data_primary')
clean_master, sku_master, _ = DataCleaner(ing.schema, ing.assumptions).clean(ing.load())
os.makedirs('data_primary/clean', exist_ok=True)
clean_master.to_parquet('data_primary/clean/clean_master.parquet', index=False)
sku_master.to_parquet('data_primary/clean/sku_master.parquet', index=False)
print('clean_master:', clean_master.shape, '| sku_master:', sku_master.shape)

## Upload Step 5a's output
Select **both** `demand_characteristics.csv` and `censoring_diagnostics.csv`.

In [ ]:
from google.colab import files
import shutil
print('Select demand_characteristics.csv AND censoring_diagnostics.csv:')
uploaded = files.upload()

demand_characteristics = pd.read_csv('demand_characteristics.csv').set_index('sku_id')
censoring_diagnostics  = pd.read_csv('censoring_diagnostics.csv').set_index('sku_id')
from src.portfolio_impact import get_flagged_skus
flagged = get_flagged_skus(censoring_diagnostics.reset_index())
for f in ('demand_characteristics.csv', 'censoring_diagnostics.csv'):
    shutil.copy(f, f'data_primary/clean/{f}')
print(f'demand_characteristics: {demand_characteristics.shape} | flagged: {len(flagged)}')

## Build the engine
Step 7 **consumes** `src/engine.py` unchanged. It does not modify it, and a test asserts the sweep leaves `run_scenario` bit-identical.

In [ ]:
import hashlib
from src.engine import TradeOffEngine, LeverSettings, build_line_master
from src.policy_model import PolicyOptimiser, PolicyModel, ATTRIBUTE_FEATURES, HISTORY_DERIVED_FEATURES, TARGETS

assumptions = yaml.safe_load(open('config/assumptions.yaml'))
schema      = yaml.safe_load(open('config/schema.yaml'))
MVD_LINE    = 'L3'

engine = TradeOffEngine(assumptions, schema, clean_master, sku_master,
                        demand_characteristics, flagged)
line_master = build_line_master(assumptions, schema)

print('engine.py sha  :', hashlib.sha256(open('src/engine.py','rb').read()).hexdigest()[:12])
print('fingerprint    :', engine.assumption_fingerprint)
print(f'{MVD_LINE}: {len(engine.line_skus(MVD_LINE))} SKUs, horizon {engine.horizon_months} months')

## Part 1 — the deterministic sweep

Grid endpoints come from `assumptions.levers.*.range`, never literals. A **grid**, not a continuous optimiser: min-run rounding and the excess-cover threshold both create steps in the cost surface, so a gradient method would chase artefacts of the discretisation.

`min_run_hours` is **not** optimised per SKU — it is a line decision, and changeovers belong to the sequence on a line, not to any one SKU. `forecast_bias_correction` is excluded as a planner-accountability lever, not a policy parameter.

In [ ]:
opt = PolicyOptimiser(engine, MVD_LINE)
cover_values, service_values = opt.default_grid(n_cover=7, n_service=6)
print(f'grid: {len(cover_values)} cover x {len(service_values)} service = '
      f'{len(cover_values)*len(service_values)} scenarios')
print('cover  :', cover_values)
print('service:', service_values)

optimal_policy = opt.optimise(cover_values, service_values)
optimal_policy.to_csv('optimal_policy.csv', index=False)

cols = ['sku_id','optimal_cover_weeks','optimal_service_target',
        'total_cost_at_default_eur','total_cost_at_optimum_eur','saving_eur','edge_optimum_flag']
print()
print(optimal_policy[cols].round(2).to_string(index=False))
print()
print(f'total saving vs line default : EUR {optimal_policy.saving_eur.sum():,.0f}')
print(f'boundary optima              : {int(optimal_policy.edge_optimum_flag.sum())} of {len(optimal_policy)}')

### Reading the boundary flag

A flagged SKU is **not** a failed search. The grid spans the *full admissible lever range* from `assumptions.levers` — a policy constraint, not a search window — so a SKU whose optimum sits at the maximum service target is telling you **the constraint binds**. The boundary is the correct operational answer, and these rows are retained for training.

Expect many of them on this line. That is the same fact **D-059** recorded: at 52% margin and a 156-week shelf life, service is a dominated choice rather than a trade-off, so most SKUs want as much of it as policy allows. On a lower-margin or shorter-life line, more interior optima should appear — a Step 8 question.

### The attribution problem — stated, not hidden

Conversion cost is a **line** property. Changeovers and capacity are shared across the SKUs on a line, so one SKU's cost is not separable from its neighbours'. Two admissible treatments:

- **`delta`** — attribute the change in line conversion cost, split across SKUs in proportion to run hours (the default)
- **`separable`** — exclude conversion cost from the per-SKU objective entirely

Neither is obviously right. Both are run below. **If the chosen optimum differs between them, that is a finding about whether the objective is separable at all** — it goes in the decision log, it does not get smoothed over.

In [ ]:
alt = opt.optimise(cover_values, service_values, mode='separable')
cmp = optimal_policy[['sku_id','optimal_cover_weeks','optimal_service_target']].merge(
    alt[['sku_id','optimal_cover_weeks','optimal_service_target']],
    on='sku_id', suffixes=('_delta','_separable'))
cmp['differs'] = ((cmp.optimal_cover_weeks_delta != cmp.optimal_cover_weeks_separable) |
                  (cmp.optimal_service_target_delta != cmp.optimal_service_target_separable))
print(cmp.to_string(index=False))
print()
n = int(cmp.differs.sum())
print(f'{n} of {len(cmp)} SKUs choose a different optimum under the two attributions.')
print('0  -> objective is separable in practice; the choice does not matter here.')
print('>0 -> log it. The per-SKU optimum depends on a modelling choice, not only on the SKU.')

## The cost surface for one SKU
Enough to see whether the minimum is a real basin or an artefact of the grid.

In [ ]:
import matplotlib.pyplot as plt
probe = optimal_policy.sku_id.iloc[0]
surf = opt.sku_grid(probe, cover_values, service_values)
piv = surf.pivot(index='service_target', columns='inventory_cover_weeks', values='total_eur')

fig, ax = plt.subplots(figsize=(8, 3.6))
im = ax.imshow(piv.values, aspect='auto', origin='lower', cmap='viridis')
ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels([f'{c:g}' for c in piv.columns])
ax.set_yticks(range(len(piv.index)));   ax.set_yticklabels([f'{i:.3f}' for i in piv.index])
ax.set_xlabel('inventory_cover_weeks'); ax.set_ylabel('service_target')
ax.set_title(f'Total attributable cost — {probe}')
plt.colorbar(im, ax=ax, label='EUR / 12 months'); plt.tight_layout(); plt.show()

## Part 2 — the policy model

Two feature sets, fitted and reported **separately**:

- **FULL** — every attribute, including the two derived from demand history
- **LAUNCH** — excludes `irreducible_volatility_cv` and `chronic_bias_l1`

LAUNCH is the set that will actually be used. Reporting only FULL would overstate real-world performance, because the most predictive features are exactly the ones a launch lacks.

**Leave-one-out, not a train/test split** — this many SKUs will not support a holdout, where one unlucky split would dominate the score. A **naive baseline** (predict the ABC-class mean) is reported alongside every model. If neither estimator beats it, that is the honest result and it is more informative than a tuned number.

In [ ]:
model = PolicyModel(optimal_policy, sku_master, demand_characteristics, line_master)
print('training rows :', len(model.training_frame()))
print('FULL features :', len(model.feature_columns('FULL')))
print('LAUNCH        :', len(model.feature_columns('LAUNCH')),
      '(excludes', HISTORY_DERIVED_FEATURES, ')')
print('seed          :', model.seed)

evaluation = model.evaluate()
evaluation.to_csv('step07_model_comparison.csv', index=False)
print()
print(evaluation.round(4).to_string(index=False))

### How to read the comparison

`beats_naive` is the column that matters, not R². With this few rows and largely categorical features, the linear benchmark may well beat XGBoost, and **neither beating the ABC-class mean is a plausible and reportable outcome** — it would say the optimal policy is not predictable from attributes alone at this sample size, which is a finding about the method's limits, not a failure to tune.

Whatever the result, the `LAUNCH` row is the honest number and `FULL` is the optimistic one.

## SHAP — why a SKU wants the policy it wants
The deliverable here is the planner-readable statement, not the plot.

In [ ]:
import shap
model.fit_final()
target = 'optimal_cover_weeks'
sv, X = model.explain(target, feature_set='LAUNCH')
shap.summary_plot(sv, X, show=True, max_display=12)

imp = (pd.DataFrame({'feature': X.columns, 'mean_abs_shap': np.abs(sv).mean(axis=0)})
       .sort_values('mean_abs_shap', ascending=False))
print(imp.head(10).round(4).to_string(index=False))

## The operational output — a new launch

No demand history, so no Step 5a characterisation and no engine simulation. Every attribute below is known before first shipment.

`predict_launch` **refuses** on a missing attribute rather than imputing one — a launch policy built on a silently-defaulted feature is worse than no recommendation, because the planner cannot see it happened. It also refuses if a history-derived feature is supplied, since that is a contradiction rather than a bonus.

In [ ]:
new_sku = {
    'category': 'personal_care',
    'abc_class': 'B',
    'gross_margin_eur': 2.10,
    'price_eur': 4.20,
    'std_cost_eur': 2.10,
    'shelf_life_days': 1092,
    'case_size': 12,
    'moq_units': 5000,
    'min_run_units': 31500,
    'line_speed_units_hr': 3500,
}
rec = model.predict_launch(new_sku)
print('RECOMMENDED OPENING POLICY')
for k, v in rec.items():
    print(f'  {k:<26}{v:>10.3f}')
print()
print('Fitted on the LAUNCH feature set. Treat as an opening position to be')
print('replaced by the engine sweep once the SKU has demand history.')

try:
    model.predict_launch({'category': 'personal_care'})
except Exception as e:
    print()
    print('Incomplete attributes ->', type(e).__name__)

## Tests

In [ ]:
sh('python -m pytest tests/test_policy_model.py -q --no-header')
sh('python -m pytest tests/test_engine.py tests/test_pipeline.py -q --no-header')

## Save outputs to Drive
`files.download()` does not complete in this environment (D-028) — Drive mount instead.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import glob, shutil
out_dir = '/content/drive/My Drive/ibp-tradeoff-outputs'
os.makedirs(out_dir, exist_ok=True)
for f in ['optimal_policy.csv', 'step07_model_comparison.csv']:
    if os.path.exists(f):
        shutil.copy(f, out_dir)
print('copied to', out_dir)

## Step 7 limitations — for the pitch, not buried in code

- **The most predictive features are the ones a launch lacks.** Volatility and chronic bias drive the optimum, and a new SKU has neither. The `LAUNCH` score is the number to quote.
- **A policy rule fitted on this many SKUs is defensible as method, not as a number.** Say so before someone asks.
- **The optimum is conditional on one assumption set.** Every row of `optimal_policy.csv` carries its fingerprint; a policy optimised under one set is not valid under another.
- **Conversion cost is not separable per SKU.** The attribution comparison above is the evidence for how much that matters here.
- **Magnitudes are not findings (§10).** Synthetic data returns whatever the generator encoded. The method is the deliverable.

**Next:** Step 8 broadens the engine to all four lines and all four levers, and the **O-10 review** follows immediately after it (D-063).

## Consolidated report — the only cell to copy

Everything above, assembled into a single output. Writes `step07_report.txt`, copies it to Drive, and prints it. **Copy this one block for review.**

Always rebuilds rather than reusing anything in memory, and prints the SHA of both modules in the header, so the provenance of every number is visible.

In [ ]:
# =============================================================================
# STEP 7 - CONSOLIDATED REPORT
# Everything for review in ONE output. Writes step07_report.txt, copies it to
# Drive, and prints it. Copy this single block.
#
# Always rebuilds. A cached object from an earlier version of src/ silently
# reports stale numbers while the test lines, which read from disk, report the
# new code - the failure mode that produced an entire invalid Step 6 report.
# =============================================================================
import os, sys, subprocess, hashlib
import numpy as np, pandas as pd, yaml

REPO = '/content/ibp-tradeoff'
os.chdir(REPO); sys.path.insert(0, REPO)

from src.ingest import DataIngestor
from src.cleaner import DataCleaner
from src.engine import TradeOffEngine, LeverSettings, build_line_master
from src.policy_model import (PolicyOptimiser, PolicyModel, optimise_all_lines,
                              ATTRIBUTE_FEATURES, HISTORY_DERIVED_FEATURES, TARGETS)
from src.portfolio_impact import get_flagged_skus

assumptions = yaml.safe_load(open('config/assumptions.yaml'))
schema      = yaml.safe_load(open('config/schema.yaml'))
MVD_LINE    = 'L3'

ing = DataIngestor(repo_root=REPO, data_root='data_primary')
clean_master, sku_master, _ = DataCleaner(ing.schema, ing.assumptions).clean(ing.load())
demand_characteristics = pd.read_csv('data_primary/clean/demand_characteristics.csv').set_index('sku_id')
censoring_diagnostics  = pd.read_csv('data_primary/clean/censoring_diagnostics.csv').set_index('sku_id')
flagged = get_flagged_skus(censoring_diagnostics.reset_index())

engine = TradeOffEngine(assumptions, schema, clean_master, sku_master,
                        demand_characteristics, flagged)
line_master = build_line_master(assumptions, schema)

# D-065: the sweep runs across ALL lines. Within one line, nine of the ten
# attribute features are constant, which forces the policy model to reduce to
# the ABC-class mean it is benchmarked against.
optimal_policy = optimise_all_lines(engine, n_cover=7, n_service=6)
optimal_policy.to_csv('optimal_policy.csv', index=False)

# O-16 comparison, MVD line only, retained as evidence for the decision
opt = PolicyOptimiser(engine, MVD_LINE)
cover_values, service_values = opt.default_grid(n_cover=7, n_service=6)
alt = opt.optimise(cover_values, service_values, mode='delta')
mvd_policy = optimal_policy[optimal_policy.line_id == MVD_LINE]

model = PolicyModel(optimal_policy, sku_master, demand_characteristics, line_master)
evaluation = model.evaluate()
evaluation.to_csv('step07_model_comparison.csv', index=False)
model.fit_final()

t_pol  = subprocess.run('python -m pytest tests/test_policy_model.py -q --no-header',
                        shell=True, capture_output=True, text=True)
t_eng  = subprocess.run('python -m pytest tests/test_engine.py -q --no-header',
                        shell=True, capture_output=True, text=True)
# test_pipeline includes a data_control regression test, so that dataset must
# exist. Building it here is cheap and deterministic.
if not os.path.isdir('data_control/raw'):
    subprocess.run('cp config/assumptions.yaml config/_bk.yaml && '
                   'cp config/assumptions_lowcensoring.yaml config/assumptions.yaml && '
                   'python generate_data.py && '
                   'cp config/_bk.yaml config/assumptions.yaml && rm config/_bk.yaml && '
                   'mv data data_control', shell=True, capture_output=True, text=True)
t_pipe = subprocess.run('python -m pytest tests/test_pipeline.py -q --no-header',
                        shell=True, capture_output=True, text=True)

L=[]; w=L.append
w('='*78); w('STEP 7 - OPTIMAL-POLICY MODEL - CONSOLIDATED REPORT'); w('='*78)
w(f'lines            : all ({len(line_master)})  |  MVD line {MVD_LINE}')
w(f'horizon          : {engine.horizon_months} months')
w(f'assumption set   : {engine.assumption_fingerprint}')
w(f'engine.py sha    : {hashlib.sha256(open("src/engine.py","rb").read()).hexdigest()[:12]}')
w(f'policy_model sha : {hashlib.sha256(open("src/policy_model.py","rb").read()).hexdigest()[:12]}')
w(f'safety stock off : {"demand (O-14 applied)" if "cv[s]) * demand_units" in open("src/engine.py").read() else "PLAN - O-14 NOT APPLIED"}')
w(f'seed             : {model.seed}')
w(f'pandas {pd.__version__} / numpy {np.__version__}')
w(f'flagged total    : {len(flagged)}  |  on {MVD_LINE}: {sorted(set(engine.line_skus(MVD_LINE)) & set(flagged))}')

w(''); w('-- 1. GRID '+'-'*66)
w(f'cover   ({len(cover_values)}): {cover_values}')
w(f'service ({len(service_values)}): {service_values}')
w(f'scenarios evaluated: {len(cover_values)*len(service_values)}')
w('Grid spans the full admissible range from assumptions.levers - a POLICY')
w('CONSTRAINT, not a search window. A boundary optimum means the constraint')
w('binds and IS the correct answer; those rows are retained for training.')

w(''); w('-- 2. OPTIMAL POLICY - ALL LINES '+'-'*45)
w(f'{len(optimal_policy)} SKUs across {optimal_policy.line_id.nunique()} lines '
  f'| attribution: separable (O-16)')
w('')
by_line = optimal_policy.groupby('line_id').agg(
    skus=('sku_id','size'),
    mean_cover=('optimal_cover_weeks','mean'),
    mean_service=('optimal_service_target','mean'),
    saving_eur=('saving_eur','sum'),
    boundary=('edge_optimum_flag','sum')).round(3)
w(by_line.to_string())
w('')
w(f'total saving vs line defaults : EUR {optimal_policy.saving_eur.sum():,.0f}')
w(f'boundary optima               : {int(optimal_policy.edge_optimum_flag.sum())} of {len(optimal_policy)}')
w(f'distinct cover values chosen  : {sorted(optimal_policy.optimal_cover_weeks.unique().tolist())}')
w(f'distinct service values       : {sorted(optimal_policy.optimal_service_target.unique().tolist())}')
w('')
w(f'--- MVD line ({MVD_LINE}) detail ---')
cols=['sku_id','optimal_cover_weeks','optimal_service_target',
      'total_cost_at_default_eur','total_cost_at_optimum_eur','saving_eur','edge_optimum_flag']
w(mvd_policy[cols].round(2).to_string(index=False))

w(''); w('-- 3. ATTRIBUTION: delta vs separable '+'-'*40)
cmp_ = mvd_policy[['sku_id','optimal_cover_weeks','optimal_service_target']].merge(
    alt[['sku_id','optimal_cover_weeks','optimal_service_target']],
    on='sku_id', suffixes=('_sep','_delta'))
cmp_['differs'] = ((cmp_.optimal_cover_weeks_delta != cmp_.optimal_cover_weeks_sep) |
                   (cmp_.optimal_service_target_delta != cmp_.optimal_service_target_sep))
w(cmp_.to_string(index=False))
n_diff = int(cmp_.differs.sum())
w('')
w(f'{n_diff} of {len(cmp_)} SKUs choose a different optimum under the two attributions.')
w('O-16 resolved: `separable` adopted. Conversion cost is a LINE property and')
w('the pro-rata attribution is one defensible split among several - a per-SKU')
w('answer that moves with that choice is not an answer. Cost of the decision,')
w('for the limitations slide: a SKU wanting small frequent batches genuinely')
w('does impose changeover cost on its neighbours, and `separable` cannot see it.')

w(''); w('-- 4. MODEL COMPARISON '+'-'*55)
w(f'training rows : {len(model.training_frame())}')
tf = model.training_frame()
var = {f: int(tf[f].nunique()) for f in ATTRIBUTE_FEATURES if f in tf.columns}
w(f'feature variance: {var}')
w('A feature with 1 distinct value cannot be split on. Fitting within a single')
w('line left only abc_class varying, forcing the model to BE the ABC-class')
w('baseline it is benchmarked against (D-065).')
w(f'FULL features : {len(model.feature_columns("FULL"))}')
w(f'LAUNCH        : {len(model.feature_columns("LAUNCH"))}  (excludes {HISTORY_DERIVED_FEATURES})')
w('')
w(evaluation.round(4).to_string(index=False))
w('')
w('`beats_naive` is the column that matters, not r2. The naive baseline is the')
w('ABC-class mean. Neither model beating it is a plausible, reportable result')
w('at this sample size - it says the optimum is not predictable from attributes')
w('alone, which is a finding about the limits of the method, not a tuning gap.')
w('LAUNCH is the honest number; FULL is the optimistic one.')

w(''); w('-- 5. SHAP - WHAT DRIVES THE POLICY '+'-'*42)
for target in TARGETS:
    try:
        sv, X = model.explain(target, feature_set='LAUNCH')
        imp = (pd.DataFrame({'feature': X.columns,
                             'mean_abs_shap': np.abs(sv).mean(axis=0)})
               .sort_values('mean_abs_shap', ascending=False).head(8))
        w(f'{target}:')
        w(imp.round(4).to_string(index=False))
        w('')
    except Exception as e:
        w(f'{target}: SHAP unavailable - {type(e).__name__}: {e}'); w('')

w('-- 6. NEW LAUNCH RECOMMENDATION '+'-'*46)
new_sku = {'category':'personal_care','abc_class':'B','gross_margin_eur':2.10,
           'price_eur':4.20,'std_cost_eur':2.10,'shelf_life_days':1092,
           'case_size':12,'moq_units':5000,'min_run_units':31500,
           'line_speed_units_hr':3500}
w('attributes: ' + ', '.join(f'{k}={v}' for k,v in new_sku.items()))
try:
    rec = model.predict_launch(new_sku)
    for k,v in rec.items():
        w(f'  {k:<28}{v:>10.3f}')
except Exception as e:
    w(f'  prediction failed: {type(e).__name__}: {e}')
try:
    model.predict_launch({'category':'personal_care'})
    w('  WARNING: incomplete attributes did NOT raise - the refusal guard is broken')
except Exception as e:
    w(f'  incomplete attributes correctly refused -> {type(e).__name__}')

w(''); w('-- 7. TESTS '+'-'*66)
for label, r in (('test_policy_model.py', t_pol), ('test_engine.py', t_eng),
                 ('test_pipeline.py', t_pipe)):
    w(f'{label:<22}: ' + (r.stdout.strip().splitlines() or ['no output'])[-1])
if any(r.returncode for r in (t_pol, t_eng, t_pipe)):
    w(''); w('FAILURES:')
    for r in (t_pol, t_eng, t_pipe):
        if r.returncode: w(r.stdout[-2500:])

w(''); w('-- 8. CHECKS '+'-'*65)
checks = [
 ('every SKU in the portfolio has a policy',
  len(optimal_policy) == len(engine.sku_master)),
 ('more than one attribute feature varies in training',
  sum(1 for f in ATTRIBUTE_FEATURES if f in tf.columns and tf[f].nunique() > 1) > 1),
 ('no optimum is worse than the line default',
  float(optimal_policy.saving_eur.min()) >= -1e-6),
 ('LAUNCH feature set excludes history-derived features',
  all(f not in model.feature_columns('LAUNCH') for f in HISTORY_DERIVED_FEATURES)),
 ('naive baseline reported for every model',
  evaluation.naive_mae.notna().all()),
 ('at least one model beats the naive baseline somewhere',
  bool(evaluation.beats_naive.any())),
 ('all test suites pass',
  all(r.returncode == 0 for r in (t_pol, t_eng, t_pipe))),
]
for label, ok in checks:
    w(f'  [{"PASS" if ok else "SEE NOTE"}]  {label}')
w('')
w('Note: "at least one model beats naive" is DIAGNOSTIC, not a gate. If it')
w('reads SEE NOTE, the result stands and gets reported as-is - it is not a')
w('reason to tune.')
w('')
w('Magnitudes are not findings (arch section 10). Every number above is')
w('whatever the generator and the assumption set encoded.')
w('='*78)

report_text = '\n'.join(L)
open('step07_report.txt','w').write(report_text)
try:
    import shutil
    d = '/content/drive/My Drive/ibp-tradeoff-outputs'
    if os.path.isdir(d):
        for f in ('step07_report.txt','optimal_policy.csv','step07_model_comparison.csv'):
            shutil.copy(f, d)
        print('saved to', d, '\n')
except Exception as e:
    print('Drive copy skipped:', e, '\n')
print(report_text)